In [ ]:
!pip install -U pip
!pip install -U accelerate bitsandbytes huggingface_hub
!pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -U pillow sentencepiece timm ftfy
!pip install -U bert-score rouge-score nltk underthesea einops
!pip install git+https://github.com/huggingface/transformers
!pip install transformers==4.57.0
!pip install --upgrade openai



Looking in indexes: https://download.pytorch.org/whl/cu121
  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-s4t6viu1
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-s4t6viu1
  Resolved https://github.com/huggingface/transformers to commit d63bb4ac4a24b2f75244cf586919b18223506a4e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.8.0.dev0-py3-none-any.whl size=11729461 sha256=e349ebc0d5b725a88a3331ce3d4c13455919cc047cdfdcc5d8d700eaf578e233
  Stored in directory: /tmp/pip-ephem-wheel-cache-li3weyw4/wheels/49/a7/50/c9fdabbf10e51bb1256adb0c1a587fedd7184f5bad28d47fe3
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled tra

In [ ]:
from openai import OpenAI
import os

os.environ["OPENAI_API_KEY"] = ""

# Khởi tạo client
client = OpenAI()  # Tự động lấy key từ env
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "Bạn là trợ lý hữu ích."},
        {"role": "user", "content": "Giới thiệu ngắn gọn về Đà Nẵng và 3 món ăn đặc sản."}
    ],
    max_tokens=300,
    temperature=0.7
)

# In kết quả
print(response.choices[0].message.content.strip())

Đà Nẵng là một thành phố nằm ở miền Trung Việt Nam, nổi tiếng với cảnh quan thiên nhiên tuyệt đẹp, bãi biển dài và sạch, cùng với sự phát triển hiện đại và năng động. Đà Nẵng còn được biết đến như một cửa ngõ quan trọng để khám phá các di sản văn hóa thế giới như Hội An và Mỹ Sơn. Thành phố này hấp dẫn du khách không chỉ bởi vẻ đẹp thiên nhiên mà còn bởi nền ẩm thực phong phú.

Ba món ăn đặc sản của Đà Nẵng mà bạn nên thử là:

1. **Mì Quảng**: Đây là món ăn biểu tượng của Đà Nẵng với sợi mì làm từ bột gạo, ăn kèm với thịt heo, tôm, gà hoặc ếch, và được chan với một lượng nước dùng vừa phải. Món ăn thường được phục vụ cùng với rau sống, đậu phộng rang, và bánh tráng mè giòn.

2. **Bánh tráng cuốn thịt heo**: Món ăn này nổi bật với sự kết hợp của thịt heo luộc thái lát mỏng, rau sống, dưa leo và các loại rau thơm, tất cả được cuốn trong bánh tráng mỏng. Đặc biệt, món này không thể thiếu nước mắm pha chế đậm đà để chấm, tạo nên hương vị đặc trưng khó


In [ ]:
from openai import OpenAI
import json
import re
from tqdm import tqdm
import pandas as pd

# ====================== CONFIGURATION ======================
jsonl_file = "Vintern_noinstruction_done.jsonl"
test_ids_file = "test_ids.csv"
output_file = jsonl_file.replace(".jsonl", "_with_geval.jsonl")

# ====================== INITIALIZE ======================
client = OpenAI()

# ====================== DEEP G-EVAL ======================
def geval_judge(question, ground_truth, prediction):
    prompt = f"""You are an expert evaluator for Visual Question Answering (VQA) on charts/tables.

Question: {question}
Ground Truth: {ground_truth}
Prediction : {prediction}

### Scoring Rules (Fine-grained):
Give score from 0.00 to 1.00 with 2-3 decimal places.

Scoring Guide:
- 1.00 : Perfect match
- 0.90-0.99 : Extremely close
- 0.80-0.89 : Very good semantic match
- 0.65-0.79 : Mostly correct
- 0.50-0.64 : Partially correct
- 0.30-0.49 : Major error
- 0.00-0.29 : Wrong

Return **ONLY** valid JSON:
{{
  "llm_judge_score": 0.92,
  "llm_judge_reasoning": "Short explanation",
  "llm_judge_match_type": "semantic"
}}

Allowed match_type: exact, semantic, partial, wrong
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "system", "content": "You are a strict, precise, and fair evaluator."},
                      {"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=400
        )
        content = response.choices[0].message.content.strip()
        match = re.search(r'\{.*\}', content, re.DOTALL)

        if match:
            result = json.loads(match.group())
            result["llm_judge_score"] = round(float(result.get("llm_judge_score", 0.0)), 3)
            return result
        else:
            return {"llm_judge_score": 0.0, "llm_judge_reasoning": "Parse error", "llm_judge_match_type": "error"}
    except:
        return {"llm_judge_score": 0.0, "llm_judge_reasoning": "Error", "llm_judge_match_type": "error"}


# ====================== MAIN ======================
def main():
    print("=== KIỂM TRA VÀ LỌC DOCUMENT_ID ===")

    # Load test_ids.csv
    df = pd.read_csv(test_ids_file)
    # Chuẩn hóa test_ids (chuyển về string, bỏ khoảng trắng, bỏ leading zero)
    test_ids = set()
    for x in df['document_id']:
        s = str(x).strip()
        test_ids.add(s)                    # giữ nguyên
        test_ids.add(s.lstrip("0"))        # bỏ leading zero
        if s.isdigit():
            test_ids.add(str(int(s)))      # dạng số thuần

    print(f"Loaded {len(test_ids)} document_ids từ CSV (sau khi chuẩn hóa)\n")

    # Load JSONL
    data = []
    with open(jsonl_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line.strip()))

    print(f"Tổng mẫu trong JSONL: {len(data)}")

    # Filter - so sánh linh hoạt
    filtered_data = []
    for item in data:
        doc_id = str(item.get("document_id", "")).strip()
        normalized = doc_id.lstrip("0")
        if doc_id in test_ids or normalized in test_ids or (doc_id.isdigit() and str(int(doc_id)) in test_ids):
            filtered_data.append(item)

    print(f"→ Sau khi lọc: {len(filtered_data)} mẫu khớp\n")

    if len(filtered_data) == 0:
        print("Vẫn không có mẫu nào khớp. Hãy kiểm tra 5 document_id đầu tiên:")
        sample_ids = set(str(item.get("document_id")).strip() for item in data[:20])
        print("Một số document_id trong JSONL:", list(sample_ids)[:10])
        return

    print("Bắt đầu chạy Deep G-Eval...\n")

    for item in tqdm(filtered_data, desc="G-Eval Judging"):
        q = item.get("question") or ""
        gt = item.get("ground_truth") or ""
        pred = item.get("prediction") or ""
        image_name = item.get("image_name", "N/A")
        doc_id = item.get("document_id", "N/A")

        judge = geval_judge(q, gt, pred)
        item.update(judge)

        score = judge.get("llm_judge_score", 0.0)
        mtype = judge.get("llm_judge_match_type", "error")
        q_short = (q[:68] + "...") if len(q) > 68 else q

        print(f"Image: {image_name} | Doc: {doc_id} | "
              f"Q: {q_short} | "
              f"Pred: {pred} | GT: {gt} | "
              f"G-Eval: {score:.3f} ({mtype})")

    # Save
    with open(output_file, 'w', encoding='utf-8') as f:
        for item in filtered_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

    print("\n" + "="*90)
    print("HOÀN THÀNH!")
    print(f"Output file: {output_file}")
    print(f"Đã đánh giá: {len(filtered_data)} mẫu")
    print("="*90)


if __name__ == "__main__":
    main()

=== KIỂM TRA VÀ LỌC DOCUMENT_ID ===
Loaded 130 document_ids từ CSV (sau khi chuẩn hóa)

Tổng mẫu trong JSONL: 7435
→ Sau khi lọc: 1105 mẫu khớp

Bắt đầu chạy Deep G-Eval...



G-Eval Judging:   0%|          | 1/1105 [00:01<36:43,  2.00s/it]

Image: 0264.jpg | Doc: 0264 | Q: Bảng trên thể hiện nội dung gì? | Pred: SỐ LƯỢNG TRÂU VÀ SẢN LƯỢNG THỊT TRÂU NƯỚC TA, GIAI ĐOẠN 2015 - 2021 | GT: Bảng trên thể hiện số lượng trâu và sản lượng thịt trâu của nước ta, giai đoạn 2015 - 2021 | G-Eval: 0.920 (semantic)


G-Eval Judging:   0%|          | 2/1105 [00:03<29:35,  1.61s/it]

Image: 0264.jpg | Doc: 0264 | Q: Năm 2015, số lượng trâu là bao nhiêu nghìn con? | Pred: 2626,1 | GT: 2626,1 nghìn con | G-Eval: 0.920 (semantic)


...

Image: 0793.PNG | Doc: 0793 | Q: Sản lượng lương thực của Việt Nam năm 1999 tăng hay giảm bao nhiêu n... | Pred: 33150 - 26142 = 7008 | GT: Tăng 7008 nghìn tấn | G-Eval: 0.920 (semantic)


G-Eval Judging: 100%|█████████▉| 1104/1105 [1:35:44<00:04,  4.42s/it]

Image: 0793.PNG | Doc: 0793 | Q: Số dân của Việt Nam năm 2003 tăng hay giảm bao nhiêu nghìn người so ... | Pred: 7512 | GT: Tăng 3872 nghìn người | G-Eval: 0.300 (wrong)


G-Eval Judging: 100%|██████████| 1105/1105 [1:35:47<00:00,  5.20s/it]

Image: 0793.PNG | Doc: 0793 | Q: Tổng sản lượng lương thực của Việt Nam trong các năm 1995, 1999 và 2... | Pred: 26142, 33150, 37706 | GT: 96998 nghìn tấn | G-Eval: 0.000 (wrong)

HOÀN THÀNH!
Output file: Vintern_noinstruction_done_with_geval.jsonl
Đã đánh giá: 1105 mẫu
